# 모델 + 데이터 분석
https://www.data.go.kr/data/15083033/fileData.do

In [ ]:
from glob import glob

file_path = r'C:\SKN_29th\git_src\머신러닝\09. 실습\data\*.csv'
files = glob(file_path)
files = files[4:]
files

In [ ]:
import pandas as pd
df = pd.read_csv(files[0])
# 불필요한 컬럼제거 - 의미가 중복된 데이터는 제거하는 것이 좋다.

In [ ]:
new_cols = ['상호명', '지점명', '상권업종대분류명', '상권업종중분류명', '상권업종소분류명', 
            '시도명', '시군구명', '행정동명', '법정동명', '도로명주소', '위도', '경도']
df[new_cols].head()

In [ ]:
# 결측치 확인
shop = df[new_cols]
shop.isnull().sum()

In [ ]:
# %pip install missingno

In [ ]:
# 한글설정
import matplotlib.pyplot as plt
plt.rc('font', family='Malgun Gothic')

In [ ]:
# 결측치 시각적으로 보기
import missingno as msno
msno.matrix(shop)

In [ ]:
fig,ax = plt.subplots(figsize=(10, 6))
plt.plot(shop['경도'],shop['위도'], 'o', markersize=1)
plt.xlabel('위도')
plt.ylabel('경도')
plt.title('매장 위치 분포')
plt.show()

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(12,6))
sns.scatterplot(data=shop,x='경도',y='위도',hue='시군구명',ax=ax)

In [ ]:
# 상권업종 대분류명이  교육과 관련된 정보
import numpy as np
edu_shop = shop[shop['상권업종대분류명']=='교육']
sns.scatterplot(edu_shop,x='경도',y='위도')

In [ ]:
# 반경 200m 내 점포수
# 동일 업종수
# 경쟁 강도(같은 업종 density)

# step1 결측치 제거
# step2 공간 Feature 생성  KDTree
# step3 모델 정의(분류 회귀)
# step4 학습
from sklearn.neighbors import BallTree
coors = np.radians(shop[['위도','경도']].values)  # 각도를 degree ~~도   -> 라디안 radian으로 변환
tree = BallTree(coors,metric='haversine')  # 단위가 radian

# 500m반경
radius = 0.5 / 6371
counts = tree.query_radius(coors,r=radius,count_only=True)
shop.loc[:,'store_density_500m']  = counts

In [ ]:
# 음식점 기준
target_category = '음식'
food_mask = (shop['상권업종대분류명'] == target_category)

food_coors = np.radians(shop.loc[food_mask,['위도','경도']].values)
food_tree = BallTree(food_coors,metric='haversine') 

# 전체 좌표 기준으로 음식점 수 계산
food_counts = food_tree.query_radius(coors, r=radius, count_only=True)
shop.loc[:,'fodd_density_500'] = food_counts

In [ ]:
# 경쟁강도
shop.loc[:,'competition_ratio'] = (shop['fodd_density_500'] / shop['store_density_500m'])
# 0에 가까울수록 경쟁적음  1에 가까울수록 경쟁이 심함

In [ ]:
indices = tree.query_radius(coors, r=radius)
diversity = [len(shop.iloc[idx_list]['상권업종중분류명'].unique()) for idx_list in indices]

In [ ]:
shop['category_deiversity_500m'] = diversity

In [ ]:
shop.loc[:,'위도':].head(2)

In [ ]:
X = shop.loc[:,'위도':]
X['score'] = (
    0.4*X['store_density_500m']  # 전체상권 활성도
    -0.3*X['competition_ratio']  # 경쟁강도(패널티)
    +0.3*X['category_deiversity_500m']  #상권의 다양성
)
y = X['score']

In [ ]:
# 주변 점포수 많음 - >사람 많음 -> 좋은 입지
X.head()

In [ ]:
from lightgbm import LGBMRegressor
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)
model.fit(X,y)

In [ ]:
# 기존 매장위치가 아니라 새로운 후보 위치를 만들어서 평가
# 새로 어디에 창업할지에 대한 정보가 없음
# 가상의 후보 위치 생성
lat_min, lat_max =  shop['위도'].min(), shop['위도'].max()
lon_min, lon_max = shop['경도'].min(), shop['경도'].max()